# CoreForge · Compare base Qwen vs trained Hindi LoRA
**Research-only evaluation.** This notebook reads your saved adapter ZIP, downloads the public Qwen base model, then generates paired responses on identical Hindi prompts. It does **not** publish the weights, change Adatacore production, or upload your ZIP to a public model hub. Requires a Colab NVIDIA GPU (free quota not guaranteed). Do not upload sensitive data.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Colab Runtime → Change runtime type → T4 GPU required.'
print('Training adapter comparison GPU:', torch.cuda.get_device_name(0))


## 1. Install compatible model libraries
This Colab-specific environment installs Unsloth, Torch and the Transformers/PEFT stack. Restart runtime only if package warnings request it; rerun cells afterwards.


In [ ]:
!pip -q install 'unsloth' 'transformers>=4.51,<5' 'peft>=0.15,<1' 'accelerate>=1.4,<2' 'safetensors>=0.5,<1'


## 2. Upload Adatacore-Hindi-AI-v0.1.zip
Your downloaded ZIP is the *LoRA adapter*, not the full base model. The notebook will validate the paths and basic manifest before loading it.


In [ ]:
from google.colab import files
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1: raise ValueError('Upload exactly one ZIP: Adatacore-Hindi-AI-v0.1.zip')
print('Received:', zip_names[0], 'bytes:', len(uploaded[zip_names[0]]))


In [ ]:
import io, json, zipfile, shutil
from pathlib import Path
root = Path('/content/coreforge-compare')
root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
    files_in_zip = archive.namelist()
    adapter_config = next((p for p in files_in_zip if p.endswith('/adapter/adapter_config.json')), None)
    report_path = next((p for p in files_in_zip if p.endswith('/training_report.json')), None)
    assert adapter_config and report_path, 'Required training report / adapter config not found in ZIP'
    base_prefix = adapter_config.removesuffix('adapter_config.json')
    assert base_prefix + 'adapter_model.safetensors' in files_in_zip, 'Missing actual trained LoRA weights'
    # Extract only the adapter files, never optimizer state, to reduce local footprint.
    adapter_dir = root / 'adapter'; adapter_dir.mkdir(exist_ok=True)
    for member in archive.infolist():
        if not member.filename.startswith(base_prefix) or member.is_dir(): continue
        relative = member.filename[len(base_prefix):]
        dest = (adapter_dir / relative).resolve()
        if adapter_dir.resolve() not in dest.parents: raise ValueError('Unsafe archive path')
        dest.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, dest.open('wb') as target: shutil.copyfileobj(source, target)
    report = json.loads(archive.read(report_path))
    adapter_meta = json.loads(archive.read(adapter_config))
assert report.get('base_model') == 'Qwen/Qwen3-0.6B', 'Unexpected base model'
assert adapter_meta.get('peft_type') == 'LORA' and adapter_meta.get('r') == 8, 'Unexpected adapter architecture'
print('Valid adapter present:',(adapter_dir / 'adapter_model.safetensors').stat().st_size,'bytes')
print('Test loss:',report.get('test_metrics', {}).get('eval_loss'))


## 3. Ask the unmodified base model
Use fixed prompts and decoding so both versions receive the same inputs. This is a qualitative smoke test, not a validated benchmark. Read the answers for fluency, factual errors and instruction following.


In [ ]:
from unsloth import FastLanguageModel
from transformers import GenerationConfig
base_model, tokenizer = FastLanguageModel.from_pretrained(model_name='Qwen/Qwen3-0.6B',max_seq_length=1024,load_in_4bit=True)
FastLanguageModel.for_inference(base_model)
prompts = [
  'ट्रांसक्रिप्शन में शब्द सटीकता (Word Accuracy) कैसे जाँची जाती है?',
  'मुझे संक्षेप में समझाइए कि WER और CER में क्या अंतर है।',
  'क्या किसी रिकॉर्डिंग को सुने बिना ट्रांसक्रिप्शन के शब्द बदलने चाहिए? कारण बताइए।',
  'हिंदी में एक विनम्र और पेशेवर फीडबैक संदेश लिखिए।',
  'अगर किसी प्रश्न का सही उत्तर न पता हो, तो आप क्या करेंगे?',
]
system='You are a helpful assistant. Answer in the language of the user. Do not invent information.'
def answer(model, question):
  text=tokenizer.apply_chat_template([{'role':'system','content':system},{'role':'user','content':question}],tokenize=False,add_generation_prompt=True,enable_thinking=False)
  batch=tokenizer(text,return_tensors='pt').to('cuda')
  with torch.inference_mode():
    out=model.generate(**batch,max_new_tokens=160,do_sample=False,pad_token_id=tokenizer.eos_token_id)
  return tokenizer.decode(out[0][batch['input_ids'].shape[1]:],skip_special_tokens=True).strip()
base_answers=[answer(base_model,p) for p in prompts]
print('Baseline tested on',len(base_answers),'prompts')


## 4. Attach the trained LoRA and compare
This uses the trained adapter on the same base model. Do not claim improvement unless the paired output and later held-out benchmarks substantiate it.


In [ ]:
from peft import PeftModel
adapted = PeftModel.from_pretrained(base_model, str(adapter_dir), is_trainable=False)
adapted.eval()
trained_answers=[answer(adapted,p) for p in prompts]
for i,(prompt,before,after) in enumerate(zip(prompts,base_answers,trained_answers),1):
  print('\n'+'='*75+'\nQUESTION',i,':',prompt)
  print('\nBASE QWEN:\n',before)
  print('\nADATACORE TRAINED:\n',after)


## 5. Save comparison output
The JSON contains evaluation prompts and generated text only, **not model weights**. This notebook deliberately does not upload anything to Hugging Face or your live website.


In [ ]:
comparison={'base_model':'Qwen/Qwen3-0.6B','adapter_type':'LoRA','training_report':report,'results':[{'prompt':p,'base':a,'trained':b} for p,a,b in zip(prompts,base_answers,trained_answers)],'status':'research_only_pending_human_evaluation'}
out='/content/coreforge-comparison-results.json'
Path(out).write_text(json.dumps(comparison,ensure_ascii=False,indent=2),encoding='utf-8')
files.download(out)
print('Saved:',out)
